# NEISS 2025 Data Preprocessing — Task #6

This notebook focuses on the assigned preprocessing work: inspecting the NEISS data, standardizing inconsistent narrative formatting, normalizing only abbreviations verified from the data, tokenizing the cleaned narratives, and validating the transformations.

The original `Narrative_1` column is never overwritten.


In [1]:
from pathlib import Path
import re
import pandas as pd
from nltk.tokenize import RegexpTokenizer

print("Environment ready.")


Environment ready.


## 1. Load the fresh NEISS 2025 dataset


In [2]:
data_path = Path("../data/raw/neiss2025.tsv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path, sep="\t", low_memory=False)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())


Dataset shape: (410201, 25)
Columns:
['CPSC_Case_Number', 'Treatment_Date', 'Age', 'Sex', 'Race', 'Other_Race', 'Hispanic', 'Body_Part', 'Diagnosis', 'Other_Diagnosis', 'Body_Part_2', 'Diagnosis_2', 'Other_Diagnosis_2', 'Disposition', 'Location', 'Fire_Involvement', 'Product_1', 'Product_2', 'Product_3', 'Alcohol', 'Drug', 'Narrative_1', 'Stratum', 'PSU', 'Weight']


,CPSC_Case_Number,Treatment_Date,Age,Sex,Race,Other_Race,Hispanic,Body_Part,Diagnosis,Other_Diagnosis,...,Fire_Involvement,Product_1,Product_2,Product_3,Alcohol,Drug,Narrative_1,Stratum,PSU,Weight
0,250102066,01/01/2025,47,2,2,NaN,2,36,59,NaN,...,0,4076,1807,0,0,0,47YOF C/O RIGHT LEG LACERATION. WAS GETTING I...,M,200,46.4246
1,250102073,01/01/2025,64,1,1,NaN,2,37,64,NaN,...,0,1842,1829,0,0,0,64YOM C/O ANKLE PAIN AND RIGHT WRIST PAIN AFTE...,M,200,46.4246
2,250102084,01/01/2025,56,1,4,NaN,2,75,62,NaN,...,0,466,4056,0,0,0,56YOM C/O HEAD INJURY. LARGE POT FELL OFF A S...,M,200,46.4246
3,250102089,01/01/2025,73,1,1,NaN,2,76,59,NaN,...,0,4057,0,0,0,0,73YOM C/O FALL AND LACERATION. TRIPPED OVER H...,M,200,46.4246
4,250102091,01/01/2025,87,1,1,NaN,2,30,71,PAIN,...,0,676,0,0,0,0,87YOM C/O FALL ON BATHMAT. LEFT SHOULDER PAIN...,M,200,46.4246


## 2. Inspect the injury narratives

Check completeness, uniqueness, length, and a reproducible sample before making transformations.


In [3]:
print("Total records:", len(df))
print("Missing narratives:", df["Narrative_1"].isna().sum())
print("Unique narratives:", df["Narrative_1"].nunique())

print("\nNarrative length statistics:")
print(df["Narrative_1"].astype("string").str.len().describe())

print("\nFive random injury narratives:")
for narrative in df["Narrative_1"].dropna().sample(n=5, random_state=42):
    print("-", narrative)


Total records: 410201
Missing narratives: 0
Unique narratives: 409099

Narrative length statistics:
count      410201.0
mean     130.781007
std       66.982294
min            24.0
25%            82.0
50%           114.0
75%           163.0
max           400.0
Name: Narrative_1, dtype: Float64

Five random injury narratives:
- 30YOF TWISTED ANKLE WALKING DOWN STEPS.  DX: ANKLE SPRAIN
- 16YOF WHO PRESENTED WITH AN EARRING STUCK IN HER EAR. DX: FOREIGN BODY OF EXTERNAL EAR.
- 16YOF     WAS PLAYING A BASKETBALL GAME WHEN PATIENT JUMPED FOR A REBOUND, FELL, AND HIT HER HEAD ON THE FLOOR    DX:   CONCUSSION, NECK PAIN
- 13MOF WAS STANDING IN STROLLER AND FELL FACE FORWARD OUT OF THE STROLLER ONTO CONCRETE AT APPROX 1600 TODAY DX: CHI, FOREHEAD LACERATION
- 2YOM WITH C/O RASH ON BACK OF NECK AFTER THE MOTHER USED TEA TREE OIL TREATMENT FOR LICE. DX: ALLERGIC REACTION MINOR


## 3. Check formatting quality

Repeated whitespace is a formatting inconsistency that can be normalized without changing narrative meaning. Duplicate narratives are inspected but are not automatically removed because separate cases can legitimately share the same wording.


In [4]:
print("Exact duplicate narratives:", df["Narrative_1"].duplicated().sum())

whitespace_only = df["Narrative_1"].str.strip().eq("").sum()
leading_trailing = df["Narrative_1"].ne(df["Narrative_1"].str.strip()).sum()
repeated_whitespace = df["Narrative_1"].str.contains(r"\s{2,}", regex=True, na=False).sum()

print("Whitespace-only narratives:", whitespace_only)
print("Narratives with leading/trailing whitespace:", leading_trailing)
print("Narratives with repeated whitespace:", repeated_whitespace)


Exact duplicate narratives: 1102
Whitespace-only narratives: 0
Narratives with leading/trailing whitespace: 0
Narratives with repeated whitespace: 91590


## 4. Normalize narrative formatting

Create a new column and preserve the raw narrative. Consecutive whitespace (spaces, tabs, or line breaks) is collapsed to one space.


In [5]:
def clean_narrative(text):
    if pd.isna(text):
        return pd.NA
    return re.sub(r"\s+", " ", text).strip()

df["Narrative_clean"] = df["Narrative_1"].apply(clean_narrative)

print(
    "Repeated whitespace remaining:",
    df["Narrative_clean"].str.contains(r"\s{2,}", regex=True, na=False).sum()
)

display(df[["Narrative_1", "Narrative_clean"]].sample(10, random_state=42))


Repeated whitespace remaining: 0


,Narrative_1,Narrative_clean
346484,30YOF TWISTED ANKLE WALKING DOWN STEPS. DX: A...,30YOF TWISTED ANKLE WALKING DOWN STEPS. DX: AN...
360523,16YOF WHO PRESENTED WITH AN EARRING STUCK IN H...,16YOF WHO PRESENTED WITH AN EARRING STUCK IN H...
355425,16YOF WAS PLAYING A BASKETBALL GAME WHEN P...,16YOF WAS PLAYING A BASKETBALL GAME WHEN PATIE...
277664,13MOF WAS STANDING IN STROLLER AND FELL FACE F...,13MOF WAS STANDING IN STROLLER AND FELL FACE F...
367530,2YOM WITH C/O RASH ON BACK OF NECK AFTER THE M...,2YOM WITH C/O RASH ON BACK OF NECK AFTER THE M...
143183,2YOM WAS WALKING IN SIBLINGS *** AND SUSTAINED...,2YOM WAS WALKING IN SIBLINGS *** AND SUSTAINED...
367021,14YOM FELL FROM HIS SKATEBOARD INJURED WRIST D...,14YOM FELL FROM HIS SKATEBOARD INJURED WRIST D...
97451,3 MOF STRING WRAPPED AROUND R 2ND TOE DX: FO...,3 MOF STRING WRAPPED AROUND R 2ND TOE DX: FORE...
393982,70YOF FELL LAST NIGHT AT HOME IN HER KITCHEN. ...,70YOF FELL LAST NIGHT AT HOME IN HER KITCHEN. ...
93614,79YOF WAS GETTING UP FROM A CHAIR WHEN HER LEF...,79YOF WAS GETTING UP FROM A CHAIR WHEN HER LEF...


## 5. Inspect common narrative conventions

The following patterns were observed in the dataset. We count them before normalization. `YOM`, `YOF`, `MOM`, `MOF`, and `DX` are retained because they are structured conventions and some strings (especially `MOM`) can have different meanings depending on context.


In [6]:
abbreviation_patterns = {
    "C/O": r"\bC/O\b",
    "PT": r"\bPT\b",
    "YOM": r"\b\d+\s*YOM\b",
    "YOF": r"\b\d+\s*YOF\b",
    "MOF": r"\b\d+\s*MOF\b",
    "MOM": r"\b\d+\s*MOM\b",
    "DX": r"\bDX\s*:"
}

for name, pattern in abbreviation_patterns.items():
    count = df["Narrative_clean"].str.contains(pattern, regex=True, na=False).sum()
    print(f"{name}: {count:,} narratives")


C/O: 39,299 narratives
PT: 45,374 narratives
YOM: 208,076 narratives
YOF: 177,393 narratives
MOF: 10,806 narratives
MOM: 13,142 narratives
DX: 305,184 narratives


## 6. Normalize verified abbreviations

Based on inspection of examples from this dataset:

- `C/O` → `COMPLAINTS OF`
- `PT` → `PATIENT`

Other abbreviations are left unchanged to avoid altering meaning.


In [7]:
def normalize_abbreviations(text):
    if pd.isna(text):
        return pd.NA

    text = re.sub(r"\bC/O\b", "COMPLAINTS OF", text)
    text = re.sub(r"\bPT\b", "PATIENT", text)
    return text

df["Narrative_normalized"] = df["Narrative_clean"].apply(normalize_abbreviations)

changed = df["Narrative_clean"].ne(df["Narrative_normalized"])
print("Narratives changed by abbreviation normalization:", changed.sum())

display(
    df.loc[changed, ["Narrative_clean", "Narrative_normalized"]]
      .sample(10, random_state=42)
)


Narratives changed by abbreviation normalization: 77288


,Narrative_clean,Narrative_normalized
198965,"18YOM C/O HAND PROBLEM. PAIN IN RIGHT HAND, RI...",18YOM COMPLAINTS OF HAND PROBLEM. PAIN IN RIGH...
326814,15 YOF C/O FINGER PAIN S/P WAS PARTICIPATING I...,15 YOF COMPLAINTS OF FINGER PAIN S/P WAS PARTI...
294927,13MOM SPILLED FLUID FROM A DEHUMIDIFIER THAT W...,13MOM SPILLED FLUID FROM A DEHUMIDIFIER THAT W...
351689,55YOM C/O A RING STUCK ON HIS FINGER. DX: FB F...,55YOM COMPLAINTS OF A RING STUCK ON HIS FINGER...
386570,17 MOM C/O INJ TO HEAD/FOREHEAD FROM TRIP/FALL...,17 MOM COMPLAINTS OF INJ TO HEAD/FOREHEAD FROM...
40046,72YOF BIBEMS AFTER A FALL. PT STATES SHE WAS C...,72YOF BIBEMS AFTER A FALL. PATIENT STATES SHE ...
141804,28YOM PT WAS BIKING ON THE BELT LINE AND ANOTH...,28YOM PATIENT WAS BIKING ON THE BELT LINE AND ...
94411,10 YOF C/O TOE BURN S/P ACCIDENTALLY SPILLED B...,10 YOF COMPLAINTS OF TOE BURN S/P ACCIDENTALLY...
350520,28 YOM C/O FINGER LACERATION S/P ACCIDENTALLY ...,28 YOM COMPLAINTS OF FINGER LACERATION S/P ACC...
139064,3YOM PRESENTS WITH AN INJURY TO HIS FRENULUM. ...,3YOM PRESENTS WITH AN INJURY TO HIS FRENULUM. ...


## 7. Tokenize normalized narratives

Tokenization prepares the text for later NLP work. Stop-word removal, stemming, and lemmatization are intentionally not applied here.


In [8]:
tokenizer = RegexpTokenizer(r"\b\w+(?:[-']\w+)*\b")

df["Narrative_tokens"] = df["Narrative_normalized"].apply(tokenizer.tokenize)

display(
    df[["Narrative_normalized", "Narrative_tokens"]]
      .sample(5, random_state=42)
)


,Narrative_normalized,Narrative_tokens
346484,30YOF TWISTED ANKLE WALKING DOWN STEPS. DX: AN...,"[30YOF, TWISTED, ANKLE, WALKING, DOWN, STEPS, ..."
360523,16YOF WHO PRESENTED WITH AN EARRING STUCK IN H...,"[16YOF, WHO, PRESENTED, WITH, AN, EARRING, STU..."
355425,16YOF WAS PLAYING A BASKETBALL GAME WHEN PATIE...,"[16YOF, WAS, PLAYING, A, BASKETBALL, GAME, WHE..."
277664,13MOF WAS STANDING IN STROLLER AND FELL FACE F...,"[13MOF, WAS, STANDING, IN, STROLLER, AND, FELL..."
367530,2YOM WITH COMPLAINTS OF RASH ON BACK OF NECK A...,"[2YOM, WITH, COMPLAINTS, OF, RASH, ON, BACK, O..."


## 8. Validate text preprocessing


In [9]:
print("Total records:", len(df))
print("Missing normalized narratives:", df["Narrative_normalized"].isna().sum())
print(
    "Repeated whitespace remaining:",
    df["Narrative_normalized"].str.contains(r"\s{2,}", regex=True, na=False).sum()
)
print(
    "C/O remaining:",
    df["Narrative_normalized"].str.contains(r"\bC/O\b", regex=True, na=False).sum()
)
print(
    "PT remaining:",
    df["Narrative_normalized"].str.contains(r"\bPT\b", regex=True, na=False).sum()
)
print("Empty token lists:", df["Narrative_tokens"].apply(len).eq(0).sum())


Total records: 410201
Missing normalized narratives: 0
Repeated whitespace remaining: 0
C/O remaining: 0
PT remaining: 0
Empty token lists: 0


## 9. Next: inspect structured variables

The next portion of Task #6 is to inspect the categorical and numerical fields before deciding whether any transformations are justified. NEISS uses coded categorical variables, so numeric-looking codes should not be scaled or otherwise treated as continuous measurements without first checking their definitions.


In [10]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nNumerical summary for Age and Weight:")
display(df[["Age", "Weight"]].describe())

categorical_cols = [
    "Sex", "Race", "Hispanic", "Body_Part", "Diagnosis",
    "Disposition", "Location", "Fire_Involvement", "Alcohol", "Drug"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index())


Data types:


,dtype
CPSC_Case_Number,int64
Treatment_Date,object
Age,int64
Sex,int64
Race,int64
Other_Race,object
Hispanic,int64
Body_Part,int64
Diagnosis,int64
Other_Diagnosis,object



Numerical summary for Age and Weight:


,Age,Weight
count,410201.000000,410201.000000
mean,48.995319,40.188622
std,49.641479,28.857603
min,0.000000,7.120000
25%,13.000000,25.546200
50%,37.000000,28.100800
75%,70.000000,51.817100
max,223.000000,113.120800



--- Sex ---
Sex
0       112
1    221547
2    188542
Name: count, dtype: int64

--- Race ---
Race
0     57714
1    245218
2     79628
3     15052
4     10141
5      1700
6       748
Name: count, dtype: int64

--- Hispanic ---
Hispanic
0     45939
1     60378
2    303884
Name: count, dtype: int64

--- Body_Part ---
Body_Part
0      5402
30    15726
31    24724
32     9141
33    12900
34    12400
35    20688
36    13993
37    19080
38     2347
75    81223
76    35475
77     5606
79    34370
80     4494
81     5164
82    14034
83    13839
84       10
85     9623
87    13366
88     8120
89     6679
92    28826
93     7156
94     5815
Name: count, dtype: int64

--- Diagnosis ---
Diagnosis
41      5092
42       309
46        83
47       105
48      2616
49       481
50       654
51      3093
52      7726
53     39034
54       789
55      6176
56      8162
57     65595
58      6434
59     58450
60      2299
61      2299
62     52684
63      2517
64     30958
65      1243
66      1365
67      

## Structured Variable Preprocessing

The structured-variable inspection showed that many integer-valued NEISS
fields are categorical codes rather than continuous numerical features.
Therefore, these fields should not be numerically scaled solely based on
their pandas data type.

Age also requires special handling because NEISS uses coded values for some
young children rather than representing every value directly as age in years.
The original Age column will therefore be preserved.

Weight is a continuous numerical variable, but it represents the NEISS
statistical sample weight rather than a patient characteristic. It will be
preserved in its original form unless the downstream modeling approach
specifically requires scaling.

For this preprocessing stage, categorical codes are preserved rather than
arbitrarily transformed.

In [17]:
import numpy as np

def standardize_age(age):
    """
    Convert NEISS Age codes to age in years.

    0       = unknown
    2-115   = age in years
    201-223 = age in months
    """
    if age == 0:
        return np.nan

    if 201 <= age <= 223:
        return (age - 200) / 12

    return float(age)


df["Age_years"] = df["Age"].apply(standardize_age)

In [18]:
## validation
print("Standardized Age:")
display(df["Age_years"].describe())

print("\nAge_years dtype:", df["Age_years"].dtype)
print("Unknown ages:", df["Age_years"].isna().sum())

Standardized Age:


count    410156.000000
mean         36.424721
std          29.047495
min           0.083333
25%          10.000000
50%          29.000000
75%          63.000000
max         109.000000
Name: Age_years, dtype: float64


Age_years dtype: float64
Unknown ages: 45


### Numerical Variables

`Age` required standardization because NEISS uses special codes for infants
under two years old. A new `Age_years` variable was created while preserving
the original `Age` field. Unknown ages were represented as missing values.

`Weight` was not scaled because it is the NEISS statistical sampling weight,
not a patient characteristic. The original `Weight` field was preserved.

In [19]:
categorical_cols = [
    "Sex",
    "Race",
    "Hispanic",
    "Body_Part",
    "Diagnosis",
    "Disposition",
    "Location",
    "Fire_Involvement",
    "Alcohol",
    "Drug"
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

print("Categorical variable data types:")
display(df[categorical_cols].dtypes.to_frame("dtype"))

Categorical variable data types:


,dtype
Sex,category
Race,category
Hispanic,category
Body_Part,category
Diagnosis,category
Disposition,category
Location,category
Fire_Involvement,category
Alcohol,category
Drug,category


## Final Preprocessing Validation

This section verifies that the preprocessing steps were applied successfully
while preserving the original NEISS data.